# Lesson 27 Lab — Disaggregated Prefill and Decode

**Puzzle:** When does moving KV state between separate Prefill and Decode workers help tail latency?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Prefill and Decode prefer different batch and compute characteristics. Separating them can isolate interference and scale phases independently, but KV transfer adds bandwidth, serialization, routing, and failure costs.


## 0. Predict before running

1. Calculate KV transfer bytes for 8K BF16 context.
2. Compare 25 and 200 Gb/s ideal transfer times.
3. Write the native evidence required for a go decision.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The single-GPU lab probes installed KV-connector/NIXL interfaces and evaluates a capacity model across context sizes and link bandwidths. No two-worker native deployment is claimed.

- KV transfer lies on the request's first-token path.
- Phase separation enables independent scaling but duplicates other resources.
- Connector availability is not a working two-node deployment.


## 2. Derive the mechanism

A Prefill worker creates KV bytes proportional to prompt tokens and model cache geometry. Before Decode can continue elsewhere, that state or a transferable representation must become available. Transfer time is approximately bytes divided by effective bandwidth plus coordination latency. Disaggregation helps only if saved queue/interference time exceeds that cost at the target reliability level.

### Mechanism at a glance

```mermaid
flowchart LR
  R["prompt request"] --> P["Prefill worker"]
  P --> K["KV blocks"]
  K --> X["connector / network transfer"]
  X --> D["Decode worker"]
  D --> O["streamed tokens"]
  P -. "phase capacity" .-> S["independent scaling"]
  D -. "phase capacity" .-> S
```

### Walk it step by step

1. **Measure phase interference.** Establish the co-located TTFT/ITL problem first.
2. **Account for KV bytes.** Derive transfer size from context and model geometry.
3. **Test the connector.** Measure application bandwidth, coordination, and failures.
4. **Compare complete systems.** Include duplicated resources and end-to-end tail latency.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 27
LESSON_TITLE = 'Disaggregated Prefill and Decode'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260839
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | co-located Prefill/Decode with interference |
| Candidate | separate workers plus KV transfer |
| Held constant | model geometry, KV dtype, context lengths, bandwidth assumptions, and coordination overhead |
| Measurements | KV bytes, ideal transfer time, break-even saved delay, connector symbols, and native deployment status |
| Evidence | `capacity-model` |

**Experiment:** Probe connector vocabulary and compute transfer break-even rows from local model geometry.


## 5. Inspect the experiment code

The model labels bandwidth as an assumption and never substitutes ideal link rate for measured application throughput. Connector imports are recorded independently.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); layers=int(cfg["num_hidden_layers"]); hidden=int(cfg["hidden_size"])
heads=int(cfg["num_attention_heads"]); kv_heads=int(cfg.get("num_key_value_heads",heads)); head_dim=int(cfg.get("head_dim",hidden//heads))
per_token=2*layers*kv_heads*head_dim*2; contexts={}
for length in (2048,8192,32768):
    size=per_token*length; contexts[str(length)]={"kv_mib":size/2**20,
        "transfer_ms_25gbps":size*8/25e9*1000+.35,"transfer_ms_200gbps":size*8/200e9*1000+.35}
connector=False; symbols=[]
for module_name in ("vllm.distributed.kv_transfer","vllm.config.kv_transfer"):
    try:
        module=importlib.import_module(module_name); connector=True
        symbols.extend(x for x in dir(module) if "Connector" in x or "Nixl" in x)
    except Exception: pass
metrics={"kv_bytes_per_token":per_token,"contexts":contexts,
 "assumptions":{"coordination_ms":.35,"bandwidth_gbps":[25,200]},"connector_probe":connector,
 "connector_symbols":sorted(set(symbols))[:30],"native_disaggregation_executed":False}
analysis=(f"BF16 KV is {per_token:,} bytes/token. An 8K prompt transfers {contexts['8192']['kv_mib']:.1f} MiB: "
          f"ideal {contexts['8192']['transfer_ms_25gbps']:.2f}/{contexts['8192']['transfer_ms_200gbps']:.2f} "
          "ms at 25/200 Gb/s including declared coordination. No two-worker run occurred.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| BF16 bytes/token | 28,672 bytes |
| 8K KV transfer | 224.000 MiB |
| 8K at 25Gb/s | 75.511928 |
| 8K at 200Gb/s | 9.745241 |
| Connector probe | yes |
| Native disaggregation | no |


## 7. Explain the result

BF16 KV is 28,672 bytes/token. An 8K prompt transfers 224.0 MiB: ideal 75.51/9.75 ms at 25/200 Gb/s including declared coordination. No two-worker run occurred.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`capacity-model`**. Measured environment facts feed explicit planning arithmetic. Assumed topology, demand, bandwidth, and reserve fields remain assumptions until a native deployment test.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 27, "title": 'Disaggregated Prefill and Decode', "environment": ENV,
    "evidence_label": 'capacity-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The break-even model identifies contexts and links worth testing; it is not evidence that disaggregated serving is faster.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 27,
  "title": "Disaggregated Prefill and Decode",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260839
  },
  "evidence_label": "capacity-model",
  "metrics": {
    "kv_bytes_per_token": 28672,
    "contexts": {
      "2048": {
        "kv_mib": 56.0,
        "transfer_ms_25gbps": 19.140481920000003,
        "transfer_ms_200gbps": 2.6988102400000002
      },
      "8192": {
        "kv_mib": 224.0,
        "transfer_ms_25gbps": 75.51192768,
        "transfer_ms_200gbps": 9.74524096
      },
      "32768": {
        "kv_mib": 896.0,
        "transfer_ms_25gbps": 300.99771072000004,
        "transfer_ms_200gbps": 37.930963840000004
      }
    },
    "assumptions": {
      "coordination_ms": 0.35,
      "bandwidth_gbps": [
        25,
        200
      ]
    },
    "conne

## 9. Make the bounded decision

> The break-even model identifies contexts and links worth testing; it is not evidence that disaggregated serving is faster.

**Acceptance/rollback:** Disaggregate only when native end-to-end p95 improves after transfer, failure recovery, duplicate capacity, and operational cost are included.

**Failure analysis:** Compression, RDMA registration, topology, cache reuse, backpressure, failures, and scheduling can dominate ideal transfer arithmetic. One GPU cannot execute both roles independently.


## 10. Extend the evidence

Deploy two workers with a supported connector, trace KV events, throttle the link, kill each role, and compare complete TTFT/ITL distributions with co-location.

The full boundary and references are in [`README.md`](README.md).
